## SPARQL Query Similarity Analysis
This notebook computes Jaccard and cosine similarities between the results of successful SPARQL queries and saves the results for further analysis.

In [ ]:
import json
import multiprocessing
from tqdm import tqdm
import ijson
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import torch

### 2. Load Successful Queries and Preprocess

Extract unique result strings from each query's results for similarity comparison.

In [ ]:
def extract_result_strings(entry):
    """
    Extracts unique result strings from a query result entry.
    """
    values = set()
    for result in entry.get("results", []):
        for k, v in result.items():
            if isinstance(v, dict) and 'value' in v:
                values.add(f"{k}:{v['value']}")
    return values

with open("successful_queries.json", "r", encoding="utf-8") as f:
    data = json.load(f)

preprocessed = [(i, entry['sparql_query'], extract_result_strings(entry)) for i, entry in enumerate(data)]

### 3. Compute Jaccard Similarities (Multiprocessing)

Find similar queries based on Jaccard similarity of result sets. Save the results to a JSON file.

In [ ]:
def process_query(args, max_similars):
    """
    Finds similar queries based on Jaccard similarity of result sets.
    Returns a dictionary with base query and its similar queries.
    """
    base_id, base_query, base_results = args
    local_similars = []
    for sim_id, sim_query, sim_results in preprocessed[base_id + 1:]:
        intersection = base_results & sim_results
        union = base_results | sim_results
        jaccard_score = len(intersection) / len(union) if union else 0
        if jaccard_score > 0.8:
            local_similars.append({
                "similar_query": sim_query.strip(),
                "jaccard_score": round(jaccard_score, 3),
                "similar_results": list(sim_results)
            })
            if len(local_similars) >= max_similars:
                break
    return {
        "base_query": base_query.strip(),
        "base_results": list(base_results),
        "similar_queries": local_similars
    } if local_similars else None

max_similars = 20
cpu_count = multiprocessing.cpu_count()
args_list = [(i, q, r) for i, (idx, q, r) in enumerate(preprocessed)]
results = []

with multiprocessing.Pool(cpu_count) as pool:
    for result in tqdm(pool.imap_unordered(
            lambda args: process_query(args, max_similars), args_list),
            total=len(args_list), desc="Computing Jaccard similarities"):
        if result:
            results.append(result)

with open("jaccard_similarities_optimized.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("JSON file written: jaccard_similarities_optimized.json")

### 4. Compute Cosine Similarities with Embeddings

For each base query and its similar queries, compute cosine similarity using embeddings. Sort similar queries by cosine similarity.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"⚡ Using device: {device}")

model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", device=device)

file_path = "jaccard_similarities_optimized.json"
output_file = "cosine_similarities_sorted.json"
results_cosine = []

with open(file_path, "r", encoding="utf-8") as f:
    total_items = sum(1 for _ in ijson.items(f, "item"))

with open(file_path, "r", encoding="utf-8") as f:
    for obj in tqdm(ijson.items(f, "item"), total=total_items, desc="Processing queries"):
        base_query = obj["base_query"]
        base_vec = model.encode(base_query, convert_to_numpy=True, device=device)
        sim_queries = [sim_obj["similar_query"] for sim_obj in obj["similar_queries"]]
        if sim_queries:
            sim_vecs = model.encode(sim_queries, convert_to_numpy=True, device=device, batch_size=32)
            sim_scores = cosine_similarity([base_vec], sim_vecs)[0]
            sim_queries_processed = []
            for sim_obj, sim_score in zip(obj["similar_queries"], sim_scores):
                sim_queries_processed.append({
                    "similar_query": sim_obj["similar_query"],
                    "jaccard_score": float(sim_obj["jaccard_score"]),
                    "cosine_score": float(sim_score)
                })
            sim_queries_processed.sort(key=lambda x: x["cosine_score"], reverse=True)
        else:
            sim_queries_processed = []
        results_cosine.append({
            "base_query": base_query,
            "similar_queries": sim_queries_processed
        })

In [ ]:
## 5. Save Cosine Similarity Results
##Save the processed cosine similarity results to a JSON file.

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(results_cosine, f, indent=2, ensure_ascii=False)

print(f"Done! Saved {len(results_cosine)} processed queries to {output_file}")

### 6. Inspect Processed Similar Queries

Display or analyze the processed similar queries for a specific entry as needed.

In [ ]:
if results_cosine:
    print(json.dumps(results_cosine[0], indent=2, ensure_ascii=False))